In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/pill_detection'
DATA_DIR = f'{BASE}/data'
TRAIN_IMG_DIR = f'{DATA_DIR}/train_images'
TRAIN_ANN_DIR = f'{DATA_DIR}/train_annotations'
TEST_IMG_DIR = f'{DATA_DIR}/test_images'
OUTPUT_DIR = f'{BASE}/outputs'
YOLO_DIR = f'{BASE}/yolo_data'
yaml_path = f'{YOLO_DIR}/data.yaml'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

!pip install ultralytics ensemble-boxes -q

import glob, json, csv
import pandas as pd
from ultralytics import YOLO
from ensemble_boxes import weighted_boxes_fusion

# category_id 매핑
json_files = glob.glob(f'{TRAIN_ANN_DIR}/**/*.json', recursive=True)
dlid_set = set()
for jf in json_files:
    with open(jf, 'r', encoding='utf-8') as f:
        data = json.load(f)
    for img_info in data['images']:
        dlid_set.add(int(img_info['dl_idx']))

dlid_to_idx = {dlid: i+1 for i, dlid in enumerate(sorted(dlid_set))}
idx_to_catid = {v: k for k, v in dlid_to_idx.items()}
print(f"매핑 완료: {len(idx_to_catid)}개")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
매핑 완료: 56개


In [ ]:
# 테스트 이미지 밝기/색상 분포 확인
import cv2, os
import numpy as np

test_imgs = os.listdir(TEST_IMG_DIR)
brightness = []
for f in test_imgs[:50]:
    img = cv2.imread(os.path.join(TEST_IMG_DIR, f))
    brightness.append(img.mean())

print(f"평균 밝기: {np.mean(brightness):.2f}")
print(f"밝기 범위: {np.min(brightness):.2f} ~ {np.max(brightness):.2f}")

평균 밝기: 123.48
밝기 범위: 106.63 ~ 162.45


In [ ]:
from ultralytics import YOLO

def predict_with_tta(model_path, model_name, conf=0.25):
    model = YOLO(model_path)
    test_imgs = sorted([f for f in os.listdir(TEST_IMG_DIR) if f.endswith('.png')])
    rows = []
    ann_id = 1

    for img_file in test_imgs:
        image_id = int(img_file.replace('.png', ''))
        img_path = os.path.join(TEST_IMG_DIR, img_file)

        results = model.predict(
            img_path,
            conf=conf,
            iou=0.5,
            augment=True,   # ← TTA
            verbose=False
        )

        for r in results:
            if r.boxes is None:
                continue
            for box in r.boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                w, h = x2 - x1, y2 - y1
                score = float(box.conf[0])
                cat_id = idx_to_catid[int(box.cls[0]) + 1]  # dl_idx 변환

                rows.append([ann_id, image_id, cat_id,
                              round(x1, 2), round(y1, 2),
                              round(w, 2), round(h, 2),
                              round(score, 4)])
                ann_id += 1

    out_path = f'{OUTPUT_DIR}/pred_{model_name}.csv'
    with open(out_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['annotation_id', 'image_id', 'category_id',
                         'bbox_x', 'bbox_y', 'bbox_w', 'bbox_h', 'score'])
        writer.writerows(rows)

    print(f"[{model_name}] {len(rows)} rows → {out_path}")
    return out_path

# 실행
pred_8s  = predict_with_tta(f'{BASE}/runs/baseline_yolov8s/weights/best.pt',  'v8s')
pred_8m  = predict_with_tta(f'{BASE}/runs/yolov8m_exp/weights/best.pt',       'v8m')
pred_11s = predict_with_tta(f'{BASE}/runs/yolo11s_exp/weights/best.pt',       'v11s')

[v8s] 3509 rows → /content/drive/MyDrive/pill_detection/outputs/pred_v8s.csv
[v8m] 3491 rows → /content/drive/MyDrive/pill_detection/outputs/pred_v8m.csv
[v11s] 3375 rows → /content/drive/MyDrive/pill_detection/outputs/pred_v11s.csv


In [ ]:
# ─── Soft-NMS 앙상블 함수 ───────────────────────────────
from ensemble_boxes import soft_nms

def run_softnms_ensemble(pred_paths, weights=None, iou_thr=0.55, sigma=0.5, thresh=0.20, out_name='softnms'):
    dfs = [pd.read_csv(p) for p in pred_paths]
    all_image_ids = sorted(set().union(*[set(df['image_id']) for df in dfs]))
    IMG_W, IMG_H = 976, 1280
    rows = []
    ann_id = 1

    for img_id in all_image_ids:
        boxes_list, scores_list, labels_list = [], [], []

        for df in dfs:
            sub = df[df['image_id'] == img_id]
            if len(sub) == 0:
                boxes_list.append([])
                scores_list.append([])
                labels_list.append([])
                continue

            bboxes = []
            for _, row in sub.iterrows():
                x1 = row['bbox_x'] / IMG_W
                y1 = row['bbox_y'] / IMG_H
                x2 = (row['bbox_x'] + row['bbox_w']) / IMG_W
                y2 = (row['bbox_y'] + row['bbox_h']) / IMG_H
                bboxes.append([
                    max(0, min(1, x1)), max(0, min(1, y1)),
                    max(0, min(1, x2)), max(0, min(1, y2))
                ])

            boxes_list.append(bboxes)
            scores_list.append(sub['score'].tolist())
            labels_list.append(sub['category_id'].tolist())

        boxes, scores, labels = soft_nms(
            boxes_list, scores_list, labels_list,
            weights=weights,
            iou_thr=iou_thr,
            sigma=sigma,
            thresh=thresh
        )

        for box, score, label in zip(boxes, scores, labels):
            x1 = box[0] * IMG_W
            y1 = box[1] * IMG_H
            w  = (box[2] - box[0]) * IMG_W
            h  = (box[3] - box[1]) * IMG_H
            rows.append([ann_id, img_id, int(label),
                         round(x1, 2), round(y1, 2),
                         round(w, 2), round(h, 2),
                         round(float(score), 4)])
            ann_id += 1

    out_path = f'{OUTPUT_DIR}/{out_name}.csv'
    with open(out_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['annotation_id', 'image_id', 'category_id',
                         'bbox_x', 'bbox_y', 'bbox_w', 'bbox_h', 'score'])
        writer.writerows(rows)

    print(f"[{out_name}] {len(rows)} rows → {out_path}")
    return out_path

# ─── 실행 ───────────────────────────────────────────────
result = run_softnms_ensemble(
    pred_paths=[pred_8s, pred_8m, pred_11s],
    weights=[1.0, 1.0, 1.0],
    iou_thr=0.55,
    sigma=0.5,
    thresh=0.20,
    out_name='softnms_best3'
)

df = pd.read_csv(result)
print(df['category_id'].head(5).tolist())

[softnms_best3] 3584 rows → /content/drive/MyDrive/pill_detection/outputs/softnms_best3.csv
[1900, 16551, 27926, 29345, 1900]


In [ ]:
df_wbf = pd.read_csv(f'{OUTPUT_DIR}/wbf_best3.csv')
print(f"WBF rows: {len(df_wbf)}")

df_snms = pd.read_csv(f'{OUTPUT_DIR}/softnms_best3.csv')
print(f"Soft-NMS rows: {len(df_snms)}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/pill_detection/outputs/wbf_best3.csv'

In [ ]:
import os
files = os.listdir(OUTPUT_DIR)
for f in sorted(files):
    print(f)

pred_v11l.csv
pred_v11m.csv
pred_v11s.csv
pred_v11s_1280.csv
pred_v11s_aug.csv
pred_v11s_augv2.csv
pred_v11s_c15.csv
pred_v8m.csv
pred_v8m_1280.csv
pred_v8m_c15.csv
pred_v8s.csv
pred_v8s_c15.csv
softnms_best3.csv
submission_baseline.csv
submission_ensemble_8s_m_11s.csv
submission_ensemble_all.csv
submission_ensemble_m_11s_11saug.csv
submission_ensemble_m_11s_aug.csv
submission_ensemble_wbf.csv
submission_yolo11s.csv
submission_yolo11s_aug.csv
submission_yolo11s_conf01.csv
submission_yolo11s_conf015.csv
submission_yolo11s_conf03.csv
submission_yolo11s_conf05.csv
submission_yolo11s_tta.csv
submission_yolo11s_tta_conf015.csv
submission_yolo11s_tta_conf03.csv
submission_yolov8m.csv
today_best3_vs_v11l.csv
today_v11l_3models_new.csv
today_v11l_4models.csv
today_v11l_4models_boost.csv
today_v11l_5models.csv
today_v11l_aug_5models.csv
today_v11l_heavy.csv
today_v11l_solo.csv
today_v11m_v11l.csv
today_v11s_v11m_v11l.csv
wbf_1280_combo.csv
wbf_1280_solo.csv
wbf_3m_plus_1280m.csv
wbf_3m_plus_128

In [ ]:
df_wbf = pd.read_csv(f'{OUTPUT_DIR}/wbf_tuned_skip20.csv')
print(f"WBF rows: {len(df_wbf)}")

df_snms = pd.read_csv(f'{OUTPUT_DIR}/softnms_best3.csv')
print(f"Soft-NMS rows: {len(df_snms)}")

WBF rows: 4160
Soft-NMS rows: 3584


In [ ]:
# ─── 외출 중 순차 학습 ───────────────────────────────────

from ultralytics import YOLO

# 1. YOLOv8l
model = YOLO('yolov8l.pt')
model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=8,
    project=f'{BASE}/runs',
    name='yolov8l_exp',
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    patience=20,
    save=True,
    device=0
)
print("✅ YOLOv8l 완료")

# 2. YOLOv9m
model = YOLO('yolov9m.pt')
model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=8,
    project=f'{BASE}/runs',
    name='yolov9m_exp',
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    patience=20,
    save=True,
    device=0
)
print("✅ YOLOv9m 완료")

# 3. YOLOv11s (lr 낮춤 + warmup 조정)
model = YOLO('yolo11s.pt')
model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    project=f'{BASE}/runs',
    name='yolo11s_lr_exp',
    optimizer='AdamW',
    lr0=0.0005,
    cos_lr=True,
    warmup_epochs=5,
    patience=20,
    save=True,
    device=0
)
print("✅ YOLOv11s 저lr 완료")

print("🎉 전체 학습 완료")

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/drive/MyDrive/pill_detection/yolo_data/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8l_exp, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, 

In [ ]:
import glob, json, csv, os
import pandas as pd
from ensemble_boxes import weighted_boxes_fusion

def run_wbf_ensemble(pred_paths, weights, iou_thr=0.55, skip_box_thr=0.20, out_name='ensemble'):
    dfs = [pd.read_csv(p) for p in pred_paths]
    all_image_ids = sorted(set().union(*[set(df['image_id']) for df in dfs]))
    IMG_W, IMG_H = 976, 1280
    rows = []
    ann_id = 1

    for img_id in all_image_ids:
        boxes_list, scores_list, labels_list = [], [], []
        for df in dfs:
            sub = df[df['image_id'] == img_id]
            if len(sub) == 0:
                boxes_list.append([]); scores_list.append([]); labels_list.append([])
                continue
            bboxes = []
            for _, row in sub.iterrows():
                x1 = row['bbox_x'] / IMG_W
                y1 = row['bbox_y'] / IMG_H
                x2 = (row['bbox_x'] + row['bbox_w']) / IMG_W
                y2 = (row['bbox_y'] + row['bbox_h']) / IMG_H
                bboxes.append([max(0,min(1,x1)), max(0,min(1,y1)),
                               max(0,min(1,x2)), max(0,min(1,y2))])
            boxes_list.append(bboxes)
            scores_list.append(sub['score'].tolist())
            labels_list.append(sub['category_id'].tolist())

        boxes, scores, labels = weighted_boxes_fusion(
            boxes_list, scores_list, labels_list,
            weights=weights, iou_thr=iou_thr, skip_box_thr=skip_box_thr)

        for box, score, label in zip(boxes, scores, labels):
            x1 = box[0] * IMG_W; y1 = box[1] * IMG_H
            w = (box[2]-box[0]) * IMG_W; h = (box[3]-box[1]) * IMG_H
            rows.append([ann_id, img_id, int(label),
                         round(x1,2), round(y1,2),
                         round(w,2), round(h,2), round(float(score),4)])
            ann_id += 1

    out_path = f'{OUTPUT_DIR}/{out_name}.csv'
    with open(out_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['annotation_id','image_id','category_id',
                         'bbox_x','bbox_y','bbox_w','bbox_h','score'])
        writer.writerows(rows)
    print(f"[{out_name}] {len(rows)} rows → {out_path}")
    return out_path

In [ ]:
# ─── 새 모델 3개 예측 ───────────────────────────────────

pred_v8l = predict_with_tta(
    f'{BASE}/runs/yolov8l_exp/weights/best.pt', 'v8l')

pred_v9m = predict_with_tta(
    f'{BASE}/runs/yolov9m_exp/weights/best.pt', 'v9m')

pred_v11s_lr = predict_with_tta(
    f'{BASE}/runs/yolo11s_lr_exp/weights/best.pt', 'v11s_lr')

# ─── 앙상블 조합 시도 ───────────────────────────────────

pred_8s  = f'{OUTPUT_DIR}/pred_v8s.csv'
pred_8m  = f'{OUTPUT_DIR}/pred_v8m.csv'
pred_11s = f'{OUTPUT_DIR}/pred_v11s.csv'
pred_8l  = f'{OUTPUT_DIR}/pred_v8l.csv'
pred_9m  = f'{OUTPUT_DIR}/pred_v9m.csv'
pred_11s_lr = f'{OUTPUT_DIR}/pred_v11s_lr.csv'

# 조합 1: 기존 최고 3모델 + v8l
run_wbf_ensemble(
    [pred_8s, pred_8m, pred_11s, pred_8l],
    weights=[1.0, 1.0, 1.0, 1.0],
    iou_thr=0.55, skip_box_thr=0.20,
    out_name='wbf_4m_v8l'
)

# 조합 2: 기존 최고 3모델 + v9m
run_wbf_ensemble(
    [pred_8s, pred_8m, pred_11s, pred_9m],
    weights=[1.0, 1.0, 1.0, 1.0],
    iou_thr=0.55, skip_box_thr=0.20,
    out_name='wbf_4m_v9m'
)

# 조합 3: 기존 최고 3모델 + v11s_lr
run_wbf_ensemble(
    [pred_8s, pred_8m, pred_11s, pred_11s_lr],
    weights=[1.0, 1.0, 1.0, 1.0],
    iou_thr=0.55, skip_box_thr=0.20,
    out_name='wbf_4m_v11slr'
)

# 조합 4: 새 모델 3개만
run_wbf_ensemble(
    [pred_8l, pred_9m, pred_11s_lr],
    weights=[1.0, 1.0, 1.0],
    iou_thr=0.55, skip_box_thr=0.20,
    out_name='wbf_new3'
)

# 조합 5: 전체 6모델
run_wbf_ensemble(
    [pred_8s, pred_8m, pred_11s, pred_8l, pred_9m, pred_11s_lr],
    weights=[1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
    iou_thr=0.55, skip_box_thr=0.20,
    out_name='wbf_6models'
)

print("✅ 전체 완료 — 5개 CSV 생성됨")

[v8l] 3369 rows → /content/drive/MyDrive/pill_detection/outputs/pred_v8l.csv
[v9m] 3444 rows → /content/drive/MyDrive/pill_detection/outputs/pred_v9m.csv
[v11s_lr] 3382 rows → /content/drive/MyDrive/pill_detection/outputs/pred_v11s_lr.csv
[wbf_4m_v8l] 4313 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_4m_v8l.csv
[wbf_4m_v9m] 4333 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_4m_v9m.csv
[wbf_4m_v11slr] 4317 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_4m_v11slr.csv
[wbf_new3] 3820 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_new3.csv
[wbf_6models] 4547 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_6models.csv
✅ 전체 완료 — 5개 CSV 생성됨
